# Netflix Movies and TV Shows – Exploratory Data Analysis (EDA)

**CodeAlpha Data Analytics Internship – Task 2**

**Objective:** Explore the Netflix Movies and TV Shows dataset to understand its structure, find data quality issues, discover trends and patterns, detect anomalies, and communicate insights with statistics and charts.

**Dataset:** [Netflix Movies and TV Shows (Kaggle, by Shivam Bansal)](https://www.kaggle.com/datasets/shivamb/netflix-shows) – file `netflix_titles.csv`

**Sections**
1. Import libraries
2. Load and inspect the dataset
3. Identify data quality issues
4. Clean the data
5. Ask meaningful questions
6. Exploratory analysis
7. Visualizations
8. Key findings
9. Conclusion

> **Before running:** download `netflix_titles.csv` from Kaggle and place it in the same folder as this notebook (or upload it to Google Colab). If your file is somewhere else, change `DATA_PATH` in Step 2.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make charts look clean and show all columns when printing tables
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

print("Libraries imported successfully.")

## Step 2: Load and Inspect the Dataset

In [ ]:
from pathlib import Path

# ---- CHANGE THIS PATH if your CSV is in a different location ----
# Local Jupyter / VS Code: keep the CSV in this project folder
# Google Colab (after uploading the file): Path("/content/netflix_titles.csv")
DATA_PATH = Path.cwd() / "netflix_titles.csv"

try:
    df_raw = pd.read_csv(DATA_PATH)   # df_raw is kept untouched as the original data
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find '{DATA_PATH}'. Download netflix_titles.csv from Kaggle "
        "(https://www.kaggle.com/datasets/shivamb/netflix-shows), place it next to this "
        "notebook (or upload it to Colab), and update DATA_PATH if needed."
    )

print(f"Dataset loaded from: {DATA_PATH}")

In [ ]:
# First five and last five rows
print("First 5 rows:")
display(df_raw.head())

print("Last 5 rows:")
display(df_raw.tail())

In [ ]:
# Size of the dataset, column names and data types
rows, cols = df_raw.shape
print(f"Number of rows   : {rows:,}")
print(f"Number of columns: {cols}")

print("\nColumn names:")
print(list(df_raw.columns))

print("\nData types:")
print(df_raw.dtypes)

In [ ]:
# Overview: non-null counts and memory usage
df_raw.info()

In [ ]:
# Summary statistics
print("Numerical columns:")
display(df_raw.describe())

print("Text (non-numerical) columns:")
display(df_raw.select_dtypes(exclude="number").describe())

**What to notice:** `release_year` is the only numerical column, so `describe()` mainly summarises that one. Most columns are text. `date_added` and `duration` are stored as text and will need conversion later. The text summary shows how many unique values each column has and its most frequent value.

## Step 3: Identify Data Quality Issues

In [ ]:
# 3.1 Missing values in every column
missing_count = df_raw.isnull().sum()
missing_pct = (missing_count / len(df_raw) * 100).round(2)

missing_table = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_pct
}).sort_values("missing_count", ascending=False)

display(missing_table)

In [ ]:
# 3.2 Duplicate records
issues = {}   # we collect every issue here so we can summarise it later

issues["full_duplicate_rows"] = int(df_raw.duplicated().sum())
issues["duplicate_show_id"] = int(df_raw["show_id"].duplicated().sum())
issues["duplicate_title_type_year"] = int(
    df_raw.duplicated(subset=["title", "type", "release_year"]).sum()
)

print(f"Fully identical rows                 : {issues['full_duplicate_rows']}")
print(f"Repeated show_id values              : {issues['duplicate_show_id']}")
print(f"Repeated (title + type + release_year): {issues['duplicate_title_type_year']}")

In [ ]:
# 3.3 Invalid, inconsistent or unusual values
print("Release year range:", df_raw["release_year"].min(), "to", df_raw["release_year"].max())
print("Unique values in 'type':", df_raw["type"].dropna().unique().tolist())

# (a) Rating column should contain ratings like 'PG-13', not durations like '74 min'
rating_text = df_raw["rating"].dropna().astype(str).str.strip()
rating_has_duration = rating_text.str.contains("min", case=False)
issues["rating_contains_duration"] = int(rating_has_duration.sum())
print(f"\nRows where 'rating' looks like a duration: {issues['rating_contains_duration']}")
if rating_has_duration.any():
    print(rating_text[rating_has_duration].value_counts())

# (b) Movies should have minutes and TV Shows should have seasons in 'duration'
has_duration = df_raw["duration"].notna()
movie_bad = (df_raw["type"] == "Movie") & has_duration & ~df_raw["duration"].str.contains("min", na=False)
tv_bad = (df_raw["type"] == "TV Show") & has_duration & ~df_raw["duration"].str.contains("Season", na=False)
issues["duration_type_mismatch"] = int((movie_bad | tv_bad).sum())
print(f"Rows where 'duration' does not match 'type': {issues['duration_type_mismatch']}")

# (c) Extra leading/trailing spaces in text cells
space_cells = 0
for col in df_raw.select_dtypes(exclude="number").columns:
    s = df_raw[col].dropna().astype(str)
    space_cells += int((s != s.str.strip()).sum())
issues["text_cells_with_extra_spaces"] = space_cells
print(f"Text cells with extra leading/trailing spaces: {space_cells}")

# (d) date_added stored as text - can every value be converted to a date?
date_text = df_raw["date_added"].dropna().str.strip()
parsed_dates = pd.to_datetime(date_text, errors="coerce", format="mixed")
issues["unparseable_dates"] = int(parsed_dates.isnull().sum())
print(f"date_added values that cannot be converted to a date: {issues['unparseable_dates']}")

# (e) Overall missing values
issues["columns_with_missing_values"] = int((missing_count > 0).sum())
issues["total_missing_cells"] = int(missing_count.sum())
print(f"\nColumns with missing values: {issues['columns_with_missing_values']}")
print(f"Total missing cells         : {issues['total_missing_cells']:,}")

**Explanation of the issues checked**
- **Missing values** are the most common problem in this dataset. Columns such as `director`, `cast` and `country` are descriptive, so a missing value usually means "not recorded", not an error.
- **Duplicates:** we checked identical rows, repeated `show_id` values, and repeated title/type/year combinations. Only *fully identical* rows are safe to remove automatically.
- **Inconsistent values:** if the `rating` column contains durations such as "74 min", the values were probably entered in the wrong column.
- **Text formats:** `date_added` and `duration` are text and must be converted before they can be analysed.

## Step 4: Clean the Data
We work on a **copy** (`df`) so the original (`df_raw`) stays unchanged.

In [ ]:
# 4.1 Create a working copy
df = df_raw.copy()
print("Working copy created. Shape:", df.shape)

**4.2 Remove extra spaces from text columns.** Spaces at the start or end of a value (for example `" United States"`) make the same value look different when grouping or counting.

In [ ]:
text_cols = df.select_dtypes(exclude="number").columns
for col in text_cols:
    df[col] = df[col].str.strip()

print("Extra spaces removed from", len(text_cols), "text columns.")

**4.3 Remove duplicate records (only if justified).** We only remove rows that are *completely identical* in every column, because those are genuine repeats. Rows that only share a title are **not** removed – two different titles can legitimately share a name.

In [ ]:
rows_before = len(df)
df = df.drop_duplicates()
rows_removed = rows_before - len(df)

print(f"Rows before: {rows_before:,} | Rows after: {len(df):,} | Duplicate rows removed: {rows_removed}")

**4.4 Fix ratings that contain durations.** If a `rating` value looks like a duration (e.g. "74 min") and the `duration` cell is empty, we move the value to `duration`. The wrong rating value is then treated as missing. This keeps useful data instead of deleting the row.

In [ ]:
rating_is_duration = df["rating"].str.contains("min", case=False, na=False)
move_mask = rating_is_duration & df["duration"].isnull()

df.loc[move_mask, "duration"] = df.loc[move_mask, "rating"]   # keep the information
df.loc[rating_is_duration, "rating"] = np.nan                  # rating itself is unknown

print("Rating values that looked like durations:", int(rating_is_duration.sum()))
print("Of these, moved into the empty 'duration' column:", int(move_mask.sum()))

**4.5 Handle missing values.**
- `director`, `cast`, `country` and `rating` are text categories. Instead of deleting those rows (which would lose many useful titles), we fill the gaps with the label **"Unknown"**. These labels are excluded from country rankings later.
- `date_added` and `duration` are **not** guessed. Inventing a date or a length would create false data, so these stay missing.

In [ ]:
for col in ["director", "cast", "country", "rating"]:
    df[col] = df[col].fillna("Unknown")

print("Filled missing values with 'Unknown' in: director, cast, country, rating")

**4.6 Convert data types.**
- `date_added` (text) → a real date using `pd.to_datetime`. Values that cannot be converted become `NaT` (missing) instead of causing an error.
- We also create helper columns: `year_added`, `decade`, and split `duration` into a number (`duration_value`) and a unit (`duration_unit`: minutes for Movies, seasons for TV Shows).

In [ ]:
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce", format="mixed")
df["year_added"] = df["date_added"].dt.year

df["decade"] = (df["release_year"] // 10) * 10

df["duration_value"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)")[0], errors="coerce")
df["duration_unit"] = df["duration"].str.extract(r"\d+\s*([A-Za-z]+)")[0]

print("Data types after conversion:")
print(df[["date_added", "year_added", "decade", "duration_value", "duration_unit"]].dtypes)

**4.7 Recheck missing values and duplicates after cleaning.**

In [ ]:
missing_after = pd.DataFrame({
    "missing_before": df_raw.isnull().sum(),
    "missing_after": df.isnull().sum()
}).fillna(0).astype(int)
missing_after = missing_after.sort_values("missing_after", ascending=False)

display(missing_after)
print("Duplicate rows remaining:", int(df.duplicated().sum()))
print("Final shape:", df.shape)

## Step 5: Ask Meaningful Questions
Each answer below is calculated directly from the cleaned dataset.

### Q1. How many Movies and TV Shows are present?

In [ ]:
type_counts = df["type"].value_counts()
type_pct = (df["type"].value_counts(normalize=True) * 100).round(2)

display(pd.DataFrame({"count": type_counts, "percent": type_pct}))

### Q2. Which release years have the highest number of titles?

In [ ]:
top_years = df["release_year"].value_counts().head(10)
display(top_years.rename("number_of_titles").to_frame())

### Q3. Which countries contribute the most titles?
A title can list several countries (for example `"United States, India"`), so we split the text and count the title once for **each** listed country. "Unknown" is excluded.

In [ ]:
country_list = df["country"].str.split(",").explode().str.strip()
country_list = country_list[(country_list != "") & (country_list != "Unknown")]

top_countries = country_list.value_counts().head(10)
display(top_countries.rename("number_of_titles").to_frame())
print("Titles with a known country:", int((df["country"] != "Unknown").sum()), "of", len(df))

### Q4. Which genres appear most frequently?
The `listed_in` column holds several genres per title separated by commas, so we split it and count each genre.

In [ ]:
genre_list = df["listed_in"].dropna().str.split(",").explode().str.strip()
genre_list = genre_list[genre_list != ""]

top_genres = genre_list.value_counts().head(10)
display(top_genres.rename("number_of_titles").to_frame())

### Q5. Which content ratings are most common?

In [ ]:
rating_counts = df["rating"].value_counts()
display(rating_counts.rename("number_of_titles").to_frame())

### Q6. How has the distribution of content changed over time?

In [ ]:
# Titles per release year, split by type
year_type = df.groupby(["release_year", "type"]).size().unstack(fill_value=0)
year_type["All titles"] = year_type.sum(axis=1)

# Titles per release decade
decade_counts = df["decade"].value_counts().sort_index()
print("Titles by release decade:")
display(decade_counts.rename("number_of_titles").to_frame())

# Titles per year they were added to Netflix
added_per_year = df["year_added"].dropna().astype(int).value_counts().sort_index()
print("Titles by year added to Netflix:")
display(added_per_year.rename("number_of_titles").to_frame())

peak_year = int(year_type["All titles"].idxmax())
print(f"Release year with the most titles: {peak_year} ({int(year_type['All titles'].max()):,} titles)")

### Q7. What unusual values or data quality issues were identified?

In [ ]:
print("Issues found in the ORIGINAL data (Step 3):")
for name, value in issues.items():
    print(f"  - {name.replace('_', ' ')}: {value:,}")

print("\nColumns with the most missing values (original data):")
display(missing_table[missing_table["missing_count"] > 0])

## Step 6: Perform Exploratory Analysis

### 6.1 Numerical variables
`release_year` is the main numerical column. `duration_value` is numerical too, but it means **minutes** for Movies and **seasons** for TV Shows, so we analyse them separately.

In [ ]:
print("release_year summary:")
display(df["release_year"].describe())

movies = df[df["type"] == "Movie"].copy()
tv_shows = df[df["type"] == "TV Show"].copy()

movie_minutes = movies["duration_value"].dropna()
tv_seasons = tv_shows["duration_value"].dropna()

print("Movie duration (minutes):")
display(movie_minutes.describe())

print("TV Show duration (number of seasons):")
display(tv_seasons.describe())

print("TV Shows by number of seasons:")
display(tv_seasons.astype(int).value_counts().sort_index().rename("number_of_shows").to_frame())

In [ ]:
# Simple readable interpretation, generated from the numbers above
print("Interpretation:")
print(f"- Titles were released between {int(df['release_year'].min())} and {int(df['release_year'].max())}; "
      f"the median release year is {int(df['release_year'].median())}.")
if len(movie_minutes) > 0:
    print(f"- Movies have a median length of {movie_minutes.median():.0f} minutes "
          f"(mean {movie_minutes.mean():.1f}).")
if len(tv_seasons) > 0:
    one_season_pct = (tv_seasons == 1).mean() * 100
    print(f"- {one_season_pct:.1f}% of TV Shows have only 1 season; the maximum is {int(tv_seasons.max())} seasons.")

### 6.2 Categorical variables

In [ ]:
for col in ["type", "rating"]:
    print(f"Share of titles by {col} (%):")
    display((df[col].value_counts(normalize=True) * 100).round(2).rename("percent").to_frame().head(8))

print("Number of different countries listed:", country_list.nunique())
print("Number of different genres listed   :", genre_list.nunique())
print("Titles with an Unknown director     :", int((df["director"] == "Unknown").sum()))
print("Titles with an Unknown cast         :", int((df["cast"] == "Unknown").sum()))

### 6.3 Relationships between variables
We look at (a) how ratings differ between Movies and TV Shows, (b) whether movie length changed with release decade, and (c) the gap between a title's release year and the year it was added to Netflix.

In [ ]:
# (a) Rating mix by type (percent within each type), for the 5 most common ratings
top5_ratings = df["rating"].value_counts().head(5).index
rating_by_type = (pd.crosstab(df["type"], df["rating"], normalize="index") * 100).round(1)
print("Rating mix by type (% of each type):")
display(rating_by_type[top5_ratings])

# (b) Movie length by release decade
print("Movie length (minutes) by release decade:")
display(movies.groupby("decade")["duration_value"].agg(["count", "median", "mean"]).round(1))

movie_corr = movies[["release_year", "duration_value"]].dropna().corr().iloc[0, 1]
print(f"Correlation between release_year and movie length: {movie_corr:.3f}")
print("(Close to 0 = almost no linear relationship; close to +1 or -1 = strong relationship.)")

In [ ]:
# (c) Gap between release year and the year added to Netflix
df["years_to_add"] = df["year_added"] - df["release_year"]

print("Years between release and being added to Netflix:")
display(df["years_to_add"].describe())

print("Interpretation:")
print(f"- Median gap: {df['years_to_add'].median():.0f} year(s). "
      f"{(df['years_to_add'] == 0).mean() * 100:.1f}% of titles were added in their release year.")

### 6.4 Anomalies and unusual observations

In [ ]:
# (a) Movies with unusually short or long durations (IQR rule)
if len(movie_minutes) > 0:
    q1, q3 = movie_minutes.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    movie_outliers = movies[(movies["duration_value"] < lower) | (movies["duration_value"] > upper)]
    print(f"Normal movie length range (IQR rule): {lower:.0f} to {upper:.0f} minutes")
    print(f"Movies outside this range: {len(movie_outliers):,}")

    print("\nLongest movies:")
    display(movies.nlargest(5, "duration_value")[["title", "release_year", "duration_value"]])
    print("Shortest movies:")
    display(movies.nsmallest(5, "duration_value")[["title", "release_year", "duration_value"]])
else:
    movie_outliers = movies.iloc[0:0]
    print("No movie durations available.")

In [ ]:
# (b) Titles that appear to be added BEFORE their release year
added_before_release = df[df["years_to_add"] < 0]
print(f"Titles added to Netflix before their release year: {len(added_before_release):,}")
if len(added_before_release) > 0:
    display(added_before_release[["title", "release_year", "date_added"]].head())

# (c) Oldest titles in the dataset
print("Oldest titles by release year:")
display(df.nsmallest(5, "release_year")[["title", "type", "release_year"]])

# (d) Very uneven categories (a few categories dominate)
print(f"Share of titles from the top country: {country_list.value_counts(normalize=True).iloc[0] * 100:.1f}% of all country mentions")

## Step 7: Create Visualizations

### Chart 1: Movies vs TV Shows

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(x=type_counts.index, y=type_counts.values, color="steelblue", ax=ax)

ax.set_title("Number of Movies vs TV Shows on Netflix", fontsize=14)
ax.set_xlabel("Content Type")
ax.set_ylabel("Number of Titles")
for container in ax.containers:
    ax.bar_label(container, fmt="%d")   # show exact counts on top of the bars

plt.tight_layout()
plt.show()

top_type = type_counts.idxmax()
print("Interpretation:")
print(f"- '{top_type}' is the larger category with {type_counts.max():,} titles "
      f"({type_pct[top_type]}% of the dataset).")
if len(type_counts) > 1:
    print(f"- The smaller category has {type_counts.min():,} titles "
          f"({type_counts.max() / type_counts.min():.2f} times fewer).")

### Chart 2: Number of titles by release year

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for column in year_type.columns:
    ax.plot(year_type.index, year_type[column], label=column,
            linewidth=2.5 if column == "All titles" else 1.5)

ax.set_title("Number of Titles by Release Year", fontsize=14)
ax.set_xlabel("Release Year")
ax.set_ylabel("Number of Titles")
ax.legend(title="Content type")
plt.tight_layout()
plt.show()

since_2010 = (df["release_year"] >= 2010).mean() * 100
print("Interpretation:")
print(f"- The most titles were released in {peak_year} ({int(year_type['All titles'].max()):,} titles).")
print(f"- {since_2010:.1f}% of all titles were released in 2010 or later.")
print(f"- The earliest release year in the data is {int(df['release_year'].min())}.")

### Chart 3: Top 10 countries by number of titles

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, orient="h", color="indianred", ax=ax)

ax.set_title("Top 10 Countries by Number of Titles", fontsize=14)
ax.set_xlabel("Number of Titles")
ax.set_ylabel("Country")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.tight_layout()
plt.show()

top3 = ", ".join(top_countries.index[:3])
print("Interpretation:")
print(f"- '{top_countries.index[0]}' contributes the most titles ({top_countries.iloc[0]:,}).")
print(f"- The top 3 countries are: {top3}.")
print(f"- Titles with an unknown country ({int((df['country'] == 'Unknown').sum()):,}) are not shown in this chart.")
print("- A title with several countries is counted once for each country.")

### Chart 4: Most common genres

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=top_genres.index, y=top_genres.values, color="seagreen", ax=ax)

ax.set_title("Top 10 Most Common Genres", fontsize=14)
ax.set_xlabel("Genre")
ax.set_ylabel("Number of Titles")
plt.xticks(rotation=45, ha="right")   # rotate long genre names so they stay readable
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.tight_layout()
plt.show()

print("Interpretation:")
print(f"- '{top_genres.index[0]}' is the most frequent genre label ({top_genres.iloc[0]:,} titles).")
print(f"- The top 3 genres are: {', '.join(top_genres.index[:3])}.")
print("- Each title can have several genres, so these counts add up to more than the number of titles.")

### Chart 5: Distribution of content ratings

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=rating_counts.index, y=rating_counts.values, color="darkorange", ax=ax)

ax.set_title("Distribution of Content Ratings", fontsize=14)
ax.set_xlabel("Content Rating")
ax.set_ylabel("Number of Titles")
plt.xticks(rotation=45, ha="right")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.tight_layout()
plt.show()

top_rating = rating_counts.index[0]
print("Interpretation:")
print(f"- '{top_rating}' is the most common rating ({rating_counts.iloc[0]:,} titles, "
      f"{rating_counts.iloc[0] / len(df) * 100:.1f}% of the dataset).")
print(f"- The three most common ratings are: {', '.join(rating_counts.index[:3])}.")
if "Unknown" in rating_counts.index:
    print(f"- {rating_counts['Unknown']:,} titles have an unknown rating (missing in the original data).")

## Step 8: Key Findings
The summary below is generated from the loaded dataset, so every number comes from your data.

In [ ]:
print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)

print("\n1. Dataset size and structure")
print(f"   - Original data: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"   - After cleaning: {df.shape[0]:,} rows x {df.shape[1]} columns (includes helper columns)")

print("\n2. Main data quality issues")
worst = missing_table[missing_table["missing_count"] > 0].head(3)
for col, row in worst.iterrows():
    print(f"   - '{col}' has {int(row['missing_count']):,} missing values ({row['missing_percent']}%)")
print(f"   - Fully duplicate rows removed: {rows_removed}")
print(f"   - Ratings that looked like durations: {issues['rating_contains_duration']}")
print(f"   - Duration/type mismatches: {issues['duration_type_mismatch']}")

print("\n3. Movies vs TV Shows")
for t, c in type_counts.items():
    print(f"   - {t}: {c:,} ({type_pct[t]}%)")

print("\n4. Release-year trends")
print(f"   - Release years range from {int(df['release_year'].min())} to {int(df['release_year'].max())}")
print(f"   - Peak release year: {peak_year} ({int(year_type['All titles'].max()):,} titles)")
print(f"   - {since_2010:.1f}% of titles were released in 2010 or later")

print("\n5. Leading countries and genres")
print(f"   - Top countries: {', '.join(top_countries.index[:3])}")
print(f"   - Top genres   : {', '.join(top_genres.index[:3])}")

print("\n6. Most common content ratings")
print(f"   - {', '.join(rating_counts.index[:3])}")

print("\n7. Anomalies")
print(f"   - Movies with unusual length (IQR rule): {len(movie_outliers):,}")
print(f"   - Titles added before their release year: {len(added_before_release):,}")
print(f"   - Titles with unknown country: {int((df['country'] == 'Unknown').sum()):,}")

## Step 9: Conclusion

This EDA showed how the Netflix catalogue is structured, how much of it is incomplete, and which patterns stand out. We checked the size and data types, found missing values and inconsistent entries, cleaned them without discarding useful rows, and then answered questions about content type, release years, countries, genres and ratings. The charts and the generated findings above summarise those results using the numbers from the dataset.

Exploratory data analysis is useful because it comes **before** any modelling or reporting: it reveals data problems early, shows which questions the data can actually answer, highlights trends and unusual values, and helps us avoid drawing wrong conclusions from messy data.

**Limitations:** The dataset is a snapshot of Netflix's catalogue, not viewing activity, so it says nothing about popularity. Countries and genres are counted per label, so a title with several of them is counted several times. The "Unknown" label stands for missing information, not a real category.